# PARF-augmented SPLM — P8 cell on A100 / H100

**Cell shape:** `parf_structural_lnD-pls-θss-θbl_shakespeare_seed{seed}` (composes Patches A–D as a single cell).

## Purpose

The P6 diagnostic on the P1 dense `structural` checkpoint surfaced two unanticipated findings:

1. **Layer 1 force imbalance** — `R(ℓ=1) ≈ 3` (V_φ pair force is 3× the V_θ external-field force at the embedding-adjacent layer).
2. **Θ_φ saturation** — `|Θ_φ| ≈ 1` in layers 2–8 (boundary saturation of the bounded-tanh value-aligner; not zero-collapse).

Both reduce to a single underlying issue: **the V_φ contribution has no per-layer scale knob**. The optimiser is forced to drive the value-aligner to its rails to compensate for the small `1/r` at deep layers, while at Layer 1 the small `‖h_t − h_s‖` (~1.6 vs ~9 at deeper layers) triggers a `1/r` blow-up that V_θ cannot offset.

**P8 composite cell** introduces four minimal-drift architectural patches, all gated by config flags:

| Patch | Flag | What it does |
| --- | --- | --- |
| **A** LN-before-distance | `--ln-before-distance` | Replace `‖h_t − h_s‖` with `‖LN(h_t) − LN(h_s)‖` inside V_φ. Equalises the radial scale across layers; kills the F-Layer1 1/r blow-up. |
| **B** per-layer V_φ scale | `--per-layer-v-phi-scale` | Learnable `s_ℓ = softplus(σ_ℓ)` per integrator layer multiplies V_φ's contribution to U. Lets the optimiser down-weight Layer 1 and up-weight middle layers. |
| **C** softsign Θ | `--theta-activation softsign` | Replace `tanh` with `softsign(x) = x/(1+|x|)`. Both bounded in [−1, 1], but softsign's gradient `1/(1+|x|)²` decays polynomially, ~1000× larger than `tanh'` at logit magnitude 5. |
| **D** bilinear Θ | `--theta-form bilinear` | Replace 3K→H→1 GELU MLP with `Θ = act(θ_t^T W θ_s + b)`. Gradient-bounded; recovers the §5.2 canonical `Θ = −sin(θ_t − θ_s)` at K=2 with W skew-symmetric. |

## Pre-registered predictions

* **R(ℓ) flatness:** with B on, `R(ℓ)` should become approximately uniform across ℓ; the optimised `s_ℓ` profile should drop at ℓ=1 relative to ℓ ∈ {3, 4, 5}.
* **Θ saturation:** with C+D on, the histogram of `Θ_φ` should re-populate the interior of [−1, 1] in deeper layers (the `tanh ±1` mass should drop by ≥ 50%).
* **PPL:** if any of these four mechanisms is causally responsible for P1's underperformance, P8 should improve val PPL relative to P1 (`structural`) at the same step budget. Magnitude of improvement is the test.

## Decision rule (post-run)

* P8 val PPL ≤ P1 val PPL **and** R(ℓ) flat **and** |Θ| < 0.95 in deep layers ⇒ commit P8 as the new structural baseline; ablate Patches A–D one-at-a-time to identify the load-bearing one.
* P8 val PPL > P1 val PPL ⇒ keep P1 as baseline; the F-Layer1 / F-Θsat findings are *symptoms*, not *causes*; reopen the failure-mode taxonomy (re-read design-doc §10.4).

Reference: `docs/PARF_Augmented_SPLM_Architecture_v2.md` §10 (tuning programme), `docs/PARF-SPLM_Path_Forward_and_Experiments.md` §9.5 (P6 + P7 + P8).

## 0. Environment / repo setup

**Colab bootstrap (automatic).** When run on Google Colab the next cell:

1. mounts your Google Drive at `/content/drive`,
2. shallow-clones (`--depth 1 --branch main`) the public `dimitarpg13/semsimula` repo into `/content/semsimula` (or refreshes it if already present),
3. symlinks the data cache (`notebooks/conservative_arch/data`) to `/content/drive/MyDrive/semsimula_parflm/data` so the GPT-2 BPE tokenisation of Tiny Shakespeare is cached on Drive (one-time cost across all sessions),
4. routes **all** training outputs (ckpt, training log, val PPL plot, P6 diagnostic, `s_ℓ` profile) to `/content/drive/MyDrive/semsimula_parflm/p8_cell/seed{SEED}/` — i.e. nothing important is written into the ephemeral `/content/semsimula` clone.

When run locally (off Colab) the notebook walks up from the CWD to find the repo root and writes to the in-repo path `parf/results/p8_cell/seed{SEED}/` as before.

Tested on Colab A100/H100 and SageMaker `ml.p4d.24xlarge` / `ml.p5.48xlarge`.

In [ ]:
# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/semsimula'                     # ephemeral source
GDRIVE_OUT_REL    = 'semsimula_parflm'                        # under /content/drive/MyDrive/
GDRIVE_SUBDIR     = 'p8_cell'                                 # subdir for THIS notebook's outputs

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    """Run a shell command, stream output, raise on non-zero exit."""
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    # ---------- (a) Mount Google Drive ----------
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    # ---------- (b) Shallow-clone (or refresh) the semsimula repo ----------
    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    # ---------- (c) Pin tokenisation cache to Drive (persists across sessions) ----------
    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()           # only succeeds if empty
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is (no Drive symlink).')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    # ---------- (d) Output root: ckpt / log / plots / diagnostic all live on Drive ----------
    RESULTS_ROOT = GDRIVE_OUT / GDRIVE_SUBDIR
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    # ---------- (e) Make sure the data_module's deps are present ----------
    _sh('pip install -q transformers huggingface_hub pyarrow')

else:
    # ---------- Local / non-Colab: locate repo root by walking up from CWD ----------
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the semsimula repo root from the notebook CWD. '
            'cd into the repo before launching the notebook.'
        )
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / GDRIVE_SUBDIR
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ===== sys.path so we can import data_module / model_parf / train_parf =====
PARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for p in (str(REPO_ROOT), str(DATA_DIR), str(PARF_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'PARF_DIR      = {PARF_DIR}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')

## 1. Disable TF32, set seeds, pick device

**Why disable TF32?** PARF threads `torch.autograd.grad` through a chained `1/r` kernel and a bounded Θ activation. Both have fragile numerics in the saturation / near-singular zones, and the ~10-bit mantissa of TF32 (vs 23-bit of FP32) is too coarse for reproducible second-order autograd. We keep cuDNN and matmul on full FP32 throughout.

**Why no AMP?** PARF's force is `−∇_h U` taken via `autograd.grad(create_graph=True)`. Mixed precision (FP16/BF16) with `GradScaler` interacts badly with second-order autograd through the soft-1/r kernel. Stay in FP32.

In [ ]:
import torch
import numpy as np

# ----- TF32 OFF (Ampere/Hopper) -----
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
# Float32 matmul precision = 'highest' (no TF32, no BF16 reduction).
torch.set_float32_matmul_precision('highest')
# Default dtype stays FP32; do NOT set float64 — PARF dynamics are FP32.

# ----- Seeds -----
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# ----- Device -----
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    name = torch.cuda.get_device_name(0)
    print(f'CUDA device: {name}  capability sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0 ({cap}); P8 cell was '
              f'designed for A100 (sm_80) or H100 (sm_90).')
    print(f'  TF32 matmul = {torch.backends.cuda.matmul.allow_tf32}  '
          f'(should be False)')
    print(f'  TF32 cuDNN  = {torch.backends.cudnn.allow_tf32}  '
          f'(should be False)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device (Apple Silicon).  TF32 toggles are CUDA-only.')
else:
    device = 'cpu'
    print('CPU only — P8 training will be very slow; smoke-test only.')
print(f'\ndevice = {device}')

## 2. Load Tiny Shakespeare; build a logfreq surprisal vector on the fly

The PARF trainer's `mass_mode='logfreq'` expects a `.npy` file with one surprisal value per vocab id (50,257 for GPT-2 BPE). The repo's bundled file lives under `sarf_mass_variant/results/logfreq_surprisal.npy` — if you are running in Colab or on a fresh checkout, we generate it from `train_ids` here.

In [ ]:
from data_module import load_tiny_shakespeare, get_batch

train_ids, val_ids = load_tiny_shakespeare()
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}')
print(f'train_ids dtype={train_ids.dtype}  '
      f'min={train_ids.min()}  max={train_ids.max()}')

VOCAB_SIZE = 50257
BUNDLED_LOGFREQ = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'sarf_mass_variant' / 'results' / 'logfreq_surprisal.npy'
DRIVE_LOGFREQ   = RESULTS_ROOT / 'logfreq_surprisal_shakespeare.npy'
if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    # Built on a previous Colab session; persists on GDrive for re-use.
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq surprisal: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq surprisal from train_ids: '
          f'len={surprisal.shape[0]}  min={surprisal.min():.3f}  '
          f'max={surprisal.max():.3f}  mean={surprisal.mean():.3f}')
    print(f'Saved to: {LOGFREQ_PATH}')

## 3. Build the P8 cell

We compose all four patches on top of the H1.5 `vh=128` cell shape (`d=128, L=8, T=128, v_hidden=128, v_depth=3, mass_mode='logfreq'`) — the same shape used by the P1 baseline and the Helmholtz Q9d AAAASSSS reference. This guarantees the cross-cell PPL comparison is apples-to-apples.

In [ ]:
from train_parf import build_config
from model_parf import PARFLM
import torch.nn.functional as F

cfg, train_cfg, tag = build_config(
    mode='shakespeare',
    logfreq_path=str(LOGFREQ_PATH),
    v_phi_kind='structural',
    fixed_gamma_arg=None,
    # ----- P8 patches -----
    ln_before_distance=True,        # Patch A
    per_layer_v_phi_scale=True,     # Patch B
    per_layer_scale_init=-3.0,      # softplus(-3) ≈ 0.05
    theta_activation='softsign',    # Patch C
    theta_form='bilinear',          # Patch D
)
full_tag = f'{tag}_shakespeare_seed{SEED}'
print(f'tag = {full_tag}')
print(f'cfg = d={cfg.d}, L={cfg.L}, T={cfg.max_len}, '
      f'v_hidden={cfg.v_hidden}, v_phi_kind={cfg.v_phi_kind!r}')
print(f'P8 flags: ln_before_distance={cfg.ln_before_distance}  '
      f'per_layer_v_phi_scale={cfg.per_layer_v_phi_scale}  '
      f'theta_activation={cfg.theta_activation!r}  '
      f'theta_form={cfg.theta_form!r}')
print(f'train_cfg = {train_cfg}')

torch.manual_seed(SEED)
model = PARFLM(cfg).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_pls = model.raw_v_phi_scale.numel() if model.raw_v_phi_scale is not None else 0
print(f'\nparams: total={n_total:,}  V_theta={n_v_theta:,}  '
      f'V_phi={n_v_phi:,}  per_layer_scale={n_pls}')
print(f'init s_ℓ = {F.softplus(model.raw_v_phi_scale).detach().tolist() if model.raw_v_phi_scale is not None else None}')

## 4. Causal-violation probe (gates training)

Confirms that the four P8 patches haven't broken the strict-causal contract. Aborts before any optimiser step on leak.

In [ ]:
from causal_probe_parf import assert_causal
assert_causal(model, vocab_size=cfg.vocab_size, T=32, seed=SEED)
print('causal probe OK — no future-position leak.')

## 5. Train (4000 steps, batch=16, T=128, AdamW(0.9, 0.95))

Same optimiser / schedule as the P1 baseline. On A100 (40 GB or 80 GB) this completes in ~25 minutes; on H100 SXM5 ~14 minutes.

In [ ]:
import math
import time
import json

BATCH = train_cfg['batch_size']
BLOCK = train_cfg['block_size']
STEPS = train_cfg['steps']
LR = train_cfg['lr']
WD = train_cfg['weight_decay']
WARMUP = train_cfg['warmup_steps']
GRAD_CLIP = train_cfg['grad_clip']
EVAL_INTERVAL = train_cfg['eval_interval']
EVAL_ITERS = train_cfg['eval_iters']
LOG_INTERVAL = train_cfg['log_interval']

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
t0 = time.time()
for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        msg = (f'step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'train_loss={loss.item():.4f}  '
               f'wall={time.time() - t0:.1f}s')
        print(msg)
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}')
        log.append({'step': step + 1, 'val_loss': val_loss, 'val_ppl': val_ppl,
                    'train_loss': loss.item()})

print(f'\nTraining done.  total wall = {time.time() - t0:.1f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 6. Save checkpoint and training log

In [ ]:
RUN_DIR = RESULTS_ROOT / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'

import dataclasses
# Save format matches train_parf.py / diagnose_v_phi_channels.py contract:
#   model_state_dict + model_cfg + variant tag.
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': full_tag,
        'step': STEPS,
        'final_val_ppl': log[-1]['val_ppl'],
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
print(f'wrote ckpt: {ckpt_path}')
print(f'wrote log : {log_path}')

## 7. Run the V_φ channel diagnostic on the trained P8 ckpt

Verifies the two pre-registered predictions: R(ℓ) flatness and the Θ-saturation drop in deeper layers. Output goes under `RUN_DIR / 'p6_diagnostic'`.

In [ ]:
import subprocess
DIAG_OUT = RUN_DIR / 'p6_diagnostic'
DIAG_OUT.mkdir(parents=True, exist_ok=True)
diag_script = PARF_DIR / 'diagnostics' / 'diagnose_v_phi_channels.py'
result = subprocess.run(
    [
        sys.executable, str(diag_script),
        '--ckpt', str(ckpt_path),
        '--out', str(DIAG_OUT),
        '--n-batches', '4',
        '--batch-size', '8',
        '--block-size', '128',
        '--device', device,
        '--seed', str(SEED),
    ],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:\n' + result.stderr[-2000:])
    raise RuntimeError(f'diagnostic exited with code {result.returncode}')

In [ ]:
# Inline-display the diagnostic plots and the per-layer s_ℓ profile.
from IPython.display import Image, display, Markdown

for fname in ('channels.png', 'gradient_ratio.png'):
    fpath = DIAG_OUT / fname
    if fpath.exists():
        display(Markdown(f'### {fname}'))
        display(Image(filename=str(fpath)))
    else:
        print(f'(missing: {fpath})')

summary_md = DIAG_OUT / 'summary.md'
if summary_md.exists():
    display(Markdown('### diagnostic summary'))
    display(Markdown(summary_md.read_text()))

In [ ]:
# Per-layer learned scale profile s_ℓ — Patch B's optimiser handle.
import matplotlib.pyplot as plt

if model.raw_v_phi_scale is not None:
    s_ell = F.softplus(model.raw_v_phi_scale).detach().cpu().numpy()
    plt.figure(figsize=(7, 3.5))
    plt.bar(np.arange(1, len(s_ell) + 1), s_ell, color='#3a6ea5')
    plt.axhline(F.softplus(torch.tensor(-3.0)).item(), color='gray',
                linestyle='--', alpha=0.6, label='init s_ℓ ≈ 0.0486')
    plt.xlabel('layer ℓ')
    plt.ylabel('learned s_ℓ = softplus(σ_ℓ)')
    plt.title(f'P8 per-layer V_φ scale after {STEPS} steps')
    plt.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'per_layer_scale.png', dpi=120)
    plt.show()
    print(f's_ℓ profile = {s_ell.tolist()}')
else:
    print('(per_layer_v_phi_scale was OFF for this run.)')

## 8. Compare to the P1 baseline

Drop the P1 final val PPL number into `BASELINES` below. The P1 dense `structural` cell on Tiny Shakespeare (4000 steps, seed 0, vh=128) is the comparison reference; P5 (`sparse_k=8`) and the all-attention sibling are also useful sanity bars.

**Decision criterion:** P8 wins if `val_ppl(P8) < val_ppl(P1)` AND the diagnostic shows R(ℓ) flat AND |Θ| < 0.95 in deep layers.

In [ ]:
BASELINES = {
    'P1 (structural)':        None,  # <-- fill in from the P1 logged run
    'P1.6 (vphi_hidden=64)':  None,
    'P5 (sparse_k=8)':        None,
    'all-attn vh=128':        None,
}
p8_ppl = log[-1]['val_ppl']
print(f'\nP8 cell ({full_tag}) final val_ppl = {p8_ppl:.2f}\n')
print(f'{"baseline":<26} {"val_ppl":>10}    {"Δ vs P8":>10}')
print('-' * 52)
for name, ppl in BASELINES.items():
    if ppl is None:
        print(f'{name:<26} {"—":>10}    {"—":>10}')
    else:
        delta = p8_ppl - ppl
        verdict = '↑ worse' if delta > 0 else '↓ better'
        print(f'{name:<26} {ppl:>10.2f}    {delta:+10.2f}  ({verdict})')

## 9. Optional: ablation of the four patches (one-at-a-time)

If P8 wins overall, re-run this notebook with **only one** of `--ln-before-distance / --per-layer-v-phi-scale / --theta-activation softsign / --theta-form bilinear` enabled at a time to identify the load-bearing patch. The CLI equivalent is:

```bash
python notebooks/conservative_arch/parf/train_parf.py \
    --mode shakespeare --seed 0 \
    --ln-before-distance       # ablation A only
```

The output tag will reflect which subset is active (e.g. `parf_structural_lnD_shakespeare_seed0`).